## module 1: evaluation
load a trained checkpoint and evaluate on the test split - metrics beyond top-1, taxonomy-aware accuracy, and visuals. works on either the 03 baseline or the 04 improved checkpoint (set CHECKPOINT below).

runs **locally or on colab** (auto-detected). note: we never stored per-photo content tags (flower/leaf/bark), so "where do errors happen" uses derivable signals - a greenness proxy (foliage-heavy vs colorful bloom), per-species image count, and resolution.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.manifold import TSNE
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# environment: local files if present, else hugging face (colab / teammates)
try:
    import google.colab  # noqa
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

USE_LOCAL = Path("../data/splits.csv").exists()
HF_REPO = "dbabnigg/botanical-vision-256"  # downscaled, Colab-friendly build
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
NUM_WORKERS = 0 if os.name == "nt" else 2

if ON_COLAB:
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception:
        print("run the 'Colab: Mount Google Drive to Server' command, then rerun")
    CKPT_DIR = "/content/drive/MyDrive/botanical-vision/checkpoints"
else:
    CKPT_DIR = "../checkpoints"

CHECKPOINT = f"{CKPT_DIR}/resnet50_improved_best.pt"   # or resnet50_baseline_best.pt
print("device:", device, "| data:", "local" if USE_LOCAL else "huggingface")

In [ ]:
ck = torch.load(CHECKPOINT, weights_only=False, map_location=device)
labels = ck["labels"]
idx2label = {i:s for i,s in enumerate(labels)}
label2idx = {s:i for i,s in enumerate(labels)}

model = models.resnet50()
model.fc = nn.Linear(model.fc.in_features, len(labels))
model.load_state_dict(ck["state_dict"])
model = model.to(device).eval()
print(len(labels), "classes |", device)

In [ ]:
# test items + taxonomy + per-species train counts, from local files or hugging face
if USE_LOCAL:
    splits = pd.read_csv("../data/splits.csv")
    tax = pd.read_csv("../data/selected_species.csv")[["species","genus","family"]]
    sp2fam = dict(zip(tax["species"], tax["family"]))
    sp2gen = dict(zip(tax["species"], tax["genus"]))
    test = splits[(splits["split"]=="test") & (splits["species"].isin(labels))].reset_index(drop=True)
    species_list = test["species"].tolist()
    train_count = splits[splits["split"]=="train"].groupby("species").size()
    def load_img(i): return Image.open(test.iloc[i]["path"]).convert("RGB")
else:
    from datasets import load_dataset
    hf = load_dataset(HF_REPO)
    keep = set(labels)
    tset = hf["test"].filter(lambda e: e["species"] in keep)
    species_list = tset["species"]
    sp2fam = {s:f for s,f in zip(hf["train"]["species"], hf["train"]["family"])}  # full label coverage
    sp2gen = {s:g for s,g in zip(hf["train"]["species"], hf["train"]["genus"])}
    train_count = pd.Series(hf["train"]["species"]).value_counts()
    def load_img(i): return tset[i]["image"].convert("RGB")

N = len(species_list)
print(N, "test images across", len(set(species_list)), "species")

In [ ]:
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
eval_tf = transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

def greenness(img):
    # fraction of green-dominant pixels - a rough foliage/leaf proxy
    a = np.asarray(img.resize((64,64)), dtype=np.int16)
    r,g,b = a[...,0],a[...,1],a[...,2]
    return float(np.mean((g>r) & (g>b)))

class EvalSet(Dataset):
    def __len__(self): return N
    def __getitem__(self, i):
        img = load_img(i)
        w,h = img.size
        return eval_tf(img), label2idx[species_list[i]], greenness(img), min(w,h), i

test_dl = DataLoader(EvalSet(), batch_size=32, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# one inference pass: predictions, top-5, confidence, penultimate embeddings, per-image signals
feats = []
hook = model.avgpool.register_forward_hook(lambda m,i,o: feats.append(o.detach().cpu().reshape(o.size(0),-1)))

y_true, y_pred, y_top5, conf, green, res, idxs, embs = [],[],[],[],[],[],[],[]
with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
    for x,y,gr,rs,ix in tqdm(test_dl, desc="inference"):
        feats.clear()
        out = model(x.to(device))
        prob = out.float().softmax(1)
        y_true += y.tolist()
        y_pred += out.argmax(1).cpu().tolist()
        y_top5 += out.topk(5,1).indices.cpu().tolist()
        conf += prob.max(1).values.cpu().tolist()
        green += gr.tolist(); res += rs.tolist(); idxs += ix.tolist()
        embs.append(feats[0])
hook.remove()

y_true=np.array(y_true); y_pred=np.array(y_pred); conf=np.array(conf)
green=np.array(green); res=np.array(res)
embs=torch.cat(embs).float().numpy()
correct = y_true==y_pred
print("done:", len(y_true), "predictions")

In [ ]:
# headline metrics - top-1 is exact-match; macro-f1 / balanced acc weight every species equally
top1 = correct.mean()
top5 = np.mean([t in p for t,p in zip(y_true, y_top5)])
macro_f1 = f1_score(y_true, y_pred, average="macro")
bal_acc = balanced_accuracy_score(y_true, y_pred)
print(f"top-1        {top1:.3f}")
print(f"top-5        {top5:.3f}")
print(f"macro-F1     {macro_f1:.3f}")
print(f"balanced acc {bal_acc:.3f}")

In [ ]:
# taxonomy-aware accuracy: how often the prediction lands in the right genus / family (>= top-1)
true_sp = [idx2label[i] for i in y_true]
pred_sp = [idx2label[i] for i in y_pred]
genus_acc = np.mean([sp2gen.get(t)==sp2gen.get(p) for t,p in zip(true_sp,pred_sp)])
family_acc = np.mean([sp2fam.get(t)==sp2fam.get(p) for t,p in zip(true_sp,pred_sp)])
print(f"exact species  {top1:.3f}")
print(f"right genus    {genus_acc:.3f}")
print(f"right family   {family_acc:.3f}")

In [ ]:
# per-species accuracy - where's the spread? which are hopeless?
per_sp = pd.DataFrame({"species":true_sp,"correct":correct}).groupby("species")["correct"].mean().sort_values()

fig,axs = plt.subplots(1,2,figsize=(12,4))
fig.set_facecolor("linen")
axs[0].hist(per_sp.values,bins=20,color="darkorange")
axs[0].set_xlabel("per-species accuracy"); axs[0].set_ylabel("# species")
axs[0].set_title("Per-Species Accuracy Spread",weight="bold")

per_sp.head(12).plot.barh(ax=axs[1],color="sienna")
axs[1].set_xlabel("accuracy"); axs[1].set_title("Hardest 12 Species",weight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# family-level confusion - species-level (NxN) is unreadable; families are legible and show real structure
true_fam = [sp2fam.get(s,"?") for s in true_sp]
pred_fam = [sp2fam.get(s,"?") for s in pred_sp]
fams = sorted(set(true_fam))
cm = pd.crosstab(pd.Series(true_fam,name="true"), pd.Series(pred_fam,name="pred")).reindex(index=fams,columns=fams,fill_value=0)
cm_norm = cm.div(cm.sum(1).replace(0,1),axis=0)

fig,ax = plt.subplots(figsize=(min(14,1+len(fams)*0.5),min(12,1+len(fams)*0.45)))
fig.set_facecolor("linen")
sns.heatmap(cm_norm,ax=ax,cmap="Oranges",cbar=True,square=True,linewidths=0.3)
ax.set_title("Family-Level Confusion (row-normalized)",weight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# most-confused species pairs - the fine-grained look-alikes it swaps
wrong = pd.DataFrame({"true":[t for t,c in zip(true_sp,correct) if not c],
                      "pred":[p for p,c in zip(pred_sp,correct) if not c]})
pairs = wrong.groupby(["true","pred"]).size().sort_values(ascending=False).head(15)
print(pairs.to_string())

In [ ]:
# where do errors happen? accuracy vs derivable signals
def acc_by_bin(values, correct, q=5):
    df = pd.DataFrame({"v":values,"c":correct})
    df["bin"] = pd.qcut(df["v"], q, duplicates="drop")
    g = df.groupby("bin",observed=True)["c"].mean()
    return [str(round(iv.mid,2)) for iv in g.index], g.values

tr_count = np.array([train_count.get(s,0) for s in true_sp])
signals = [("greenness (leaf proxy)",green),("train images / species",tr_count),("resolution (min side)",res)]

fig,axs = plt.subplots(1,3,figsize=(15,4))
fig.set_facecolor("linen")
for ax,(name,vals) in zip(axs,signals):
    xs,ys = acc_by_bin(vals,correct)
    ax.bar(xs,ys,color="gold",edgecolor="sienna")
    ax.axhline(top1,ls="--",color="sienna",lw=1)
    ax.set_title(name,weight="bold"); ax.set_ylabel("accuracy"); ax.tick_params(axis="x",rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# prediction grid - green title = correct, red = wrong; shows true -> pred (confidence)
rng = np.random.default_rng(0)
pick = np.concatenate([rng.choice(np.where(correct)[0],6,replace=False),
                       rng.choice(np.where(~correct)[0],6,replace=False)])
fig,axs = plt.subplots(3,4,figsize=(13,10)); fig.set_facecolor("linen")
for ax,j in zip(axs.ravel(),pick):
    ax.imshow(load_img(idxs[j])); ax.axis("off")
    t,p = idx2label[y_true[j]], idx2label[y_pred[j]]
    ax.set_title(f"{t}\n-> {p} ({conf[j]:.2f})", fontsize=8,
                 color="green" if correct[j] else "firebrick")
plt.tight_layout(); plt.show()

In [ ]:
# do the learned features cluster by family? t-sne on a sample of the 2048-dim embeddings
n = min(1500, len(embs))
sel = np.random.default_rng(0).choice(len(embs), n, replace=False)
emb2d = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(embs[sel])

fam_sel = np.array(true_fam)[sel]
top_fams = pd.Series(fam_sel).value_counts().head(10).index
fig,ax = plt.subplots(figsize=(9,7)); fig.set_facecolor("linen")
for fam,color in zip(top_fams, sns.color_palette("husl",len(top_fams))):
    m = fam_sel==fam
    ax.scatter(emb2d[m,0],emb2d[m,1],s=10,color=color,label=fam,alpha=0.7)
ax.legend(fontsize=7,markerscale=1.5,loc="best"); ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Embeddings by Family (t-SNE)",weight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# grad-cam - where does the model look? (should be on the plant, not the background)
acts, grads = {}, {}
f = model.layer4.register_forward_hook(lambda m,i,o: acts.__setitem__("v", o.detach()))
b = model.layer4.register_full_backward_hook(lambda m,gi,go: grads.__setitem__("v", go[0].detach()))

def gradcam(pos):
    img = load_img(pos)
    x = eval_tf(img).unsqueeze(0).to(device)
    out = model(x); out[0, out.argmax()].backward()
    w = grads["v"].mean((2,3), keepdim=True)
    cam = (w*acts["v"]).sum(1).relu()[0].cpu().numpy()
    cam = (cam-cam.min())/(np.ptp(cam)+1e-8)
    return img, cam

fig,axs = plt.subplots(2,4,figsize=(13,6)); fig.set_facecolor("linen")
for k,j in enumerate(pick[:4]):
    img,cam = gradcam(idxs[j])
    axs[0,k].imshow(img); axs[0,k].axis("off"); axs[0,k].set_title(idx2label[y_true[j]],fontsize=8)
    axs[1,k].imshow(img); axs[1,k].imshow(np.asarray(Image.fromarray((cam*255).astype(np.uint8)).resize(img.size)),cmap="jet",alpha=0.5); axs[1,k].axis("off")
f.remove(); b.remove()
plt.tight_layout(); plt.show()